***

## Preparing Workspace

***

In [ ]:
import pandas as pd
import os
import numpy as np
from functools import partial
import re
import glob
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio


***

## **Translating**

***

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'IHME')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'IHME Data')
    path_main = os.path.join(path_sp, 'Data')

path_config0 = os.path.join(path_git, 'config')
path_code    = os.path.join(path_git, 'Pipelines', 'IHME')

In [ ]:
# Defining sacog list for subsetting

sacog = ["El Dorado", "Placer", "Sacramento", "Sutter", "Yolo", "Yuba"]
sacog = [f"{county} County (California)" for county in sacog]

In [ ]:
# Should be noted, the entire CountyLifeExp folder from Seth's file in Process Revamp > Task 8 needs to be downloaded for this to work. 

os.chdir(path_raw)

# Glob just finds all the csvs in a folder 

files = glob.glob('*.csv')

df_ihme = pd.concat([pd.read_csv(file) for file in files]).reset_index()

# Selecting the needed cols
df_ihme = df_ihme[['year', 'location_name', 'race_name', 'age_name', 'val', 'upper', 'lower']]
df_ihme['location_name'] = df_ihme['location_name'].str.replace(" County \\(California\\)", "")
df_ihme.columns = ["Year", "County", "Race", "Age", "Estimate", "Upper", "Lower"]

# Just making it into a pd df and subsetting
df_ihme = pd.DataFrame(df_ihme)
df_ihme = df_ihme[df_ihme['County'].isin(sacog)]


display(df_ihme.head())

In [ ]:
# Calculating mean for specific cols and creating the total dfs on county, race and year
numeric_columns = ['Estimate', 'Upper', 'Lower']
df_counties = df_ihme[df_ihme['Age'] == "<1 year"].groupby(['Year', 'County', 'Race'])[numeric_columns].mean().reset_index()
df_counties['Age'] = "All Ages Total"
df_counties = df_counties[['Year', 'County', 'Race', 'Age'] + numeric_columns]  # Reorder columns

df_mpo = df_ihme[df_ihme['Age'] == "<1 year"].groupby(['Year', 'Race'])[numeric_columns].mean().reset_index()
df_mpo['MPO'] = "SACOG Total"
df_mpo['Age'] = "All Ages Total"
df_mpo = df_mpo[['Year', 'MPO', 'Race', 'Age'] + numeric_columns]  # Reorder columns




df_ihme['Race_sort'] = pd.Categorical(df_ihme['Race'], ['Total'
                                                         , 'AIAN'
                                                         , 'API'
                                                         , 'Black'
                                                         , 'Latino'
                                                         , 'White'])
df_ihme = df_ihme.sort_values(['County', 'Year', 'Race_sort'], ascending = [True, False, True])
df_ihme = df_ihme.drop(['Race_sort'], axis = 1)
df_ihme = df_ihme.reset_index(drop = True)

df_counties['Race_sort'] = pd.Categorical(df_counties['Race'], ['Total'
                                                                 , 'AIAN'
                                                                 , 'API'
                                                                 , 'Black'
                                                                 , 'Latino'
                                                                 , 'White'])
df_counties = df_counties.sort_values(['County', 'Year', 'Race_sort'], ascending = [True, False, True])
df_counties = df_counties.drop(['Race_sort'], axis = 1)
df_counties = df_counties.reset_index(drop = True)


df_mpo['Race_sort'] = pd.Categorical(df_mpo['Race'], ['Total'
                                                       , 'AIAN'
                                                       , 'API'
                                                       , 'Black'
                                                       , 'Latino'
                                                       , 'White'])
df_mpo = df_mpo.sort_values(['MPO', 'Year', 'Race_sort'], ascending = [True, False, True])
df_mpo = df_mpo.drop(['Race_sort'], axis = 1)
df_mpo = df_mpo.reset_index(drop = True)

In [ ]:
# Displaying

display(df_ihme.head(10))
display(df_counties.head(10))
display(df_mpo.head(10))

Should be noted, some of the calculations for estimates, upper, and lower are slightly different compared to Seth's values. For instance, I had an estimate of 80.431622 for El Dorado County (California) AIAN in 2000. Conversely, Seth had 80.1633875910602. Another example, for El Dorado API 2000, I had 83.191716 and Seth had 83.19933999842. 


Josh - this might be because of rounding differences between R and Python.  R has a weird bias towards even numbers when rounding.  So in R, 1.5 and 2.5 both round to a whole number of 2, instead of 2 and 3 respectively.

In [ ]:
# path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Healthy Places', 'Health_4 Life Expectancy')


# with pd.ExcelWriter(os.path.join(path_out, 'Health_4 Counties by Age Groups IHME.xlsx'), engine='xlsxwriter') as writer:
# # with pd.ExcelWriter(os.path.join(path_out, 'Health_4 Counties by Age Groups IHME.xlsx'), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
#     df_ihme.to_excel(writer, index = False, sheet_name = 'Counties')


# with pd.ExcelWriter(os.path.join(path_out, 'Health_4 Counties IHME.xlsx'), engine='xlsxwriter') as writer:
# # with pd.ExcelWriter(os.path.join(path_out, 'Health_4 Counties IHME.xlsx'), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
#     df_counties.to_excel(writer, index = False, sheet_name = 'Counties')


# with pd.ExcelWriter(os.path.join(path_out, 'Health_4 MPO IHME.xlsx'), engine='xlsxwriter') as writer:
# # with pd.ExcelWriter(os.path.join(path_out, 'Health_4 MPO IHME.xlsx'), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
#     df_mpo.to_excel(writer, index = False, sheet_name = 'MPO')


In [ ]:
# df_counties.columns = [col.lower() for col in df_counties.columns]
# df_counties = df_counties[df_counties['race'] != 'Total']
# df_counties.to_csv(os.path.join(path_agol, 'Health_4_Counties_IHME.csv'), index = False)

# df_mpo.columns = [col.lower() for col in df_mpo.columns]
# df_mpo = df_mpo[df_mpo['race'] != 'Total']
# df_mpo.to_csv(os.path.join(path_agol, 'Health_4_MPO_IHME.csv'), index = False)

In [ ]:
path_plots = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Healthy Places', 'Health_4 Life Expectancy', 'plots')

df_plot = df_mpo.copy()
df_plot = df_plot[df_plot['race'] != 'Total']

x = 'year'
y = 'estimate'
color = 'race'
labels = 'race'

fig = px.line(df_plot
                 , x = x
                 , y = y
                 , color = color
                 # , line_dash = line_dash
                 , markers = False
                 , labels = labels
                )

fig.update_layout(title = 'Life Expectancy by Race (SACOG)')

fig.add_trace(go.Scatter(x=df_plot.year, y = df_plot.lower,
                         mode = 'lines', 
                         line = dict(dash='dash', color = df_plot.race, width = 1),
                         name = 'Lower Estimate'))

fig.add_trace(go.Scatter(x=df_plot.year, y = df_plot.upper,
                         mode = 'lines', 
                         line = dict(dash='dash', color = df_plot.race, width = 1),
                         name = 'Upper Estimate'))

# fig.write_html(
#     os.path.join(
#         path_plots
#         , ''.join(['Health_4_'
#                    , 'Life Expectancy by Race SACOG_'
#                    , 'line_'
#                    , '.html'])
#     )
# )
    
fig.show()